# WS-03 — Gán nhãn yếu v2 trên Google Colab

Notebook này chạy trên **Google Colab** cho bước 3 của quy trình Weak Supervision phiên bản v2:

1. Kết nối Drive và lấy đúng branch `WS`.
2. Cài phụ thuộc Colab tối thiểu của RAD.
3. Đọc `configs/ws.yaml` để xác thực dữ liệu SkinCAP đã được kiểm chứng; đường dẫn WS_03 v2 được khai báo cục bộ trong notebook.
4. Chỉ đọc và kiểm tra artifact đã chốt của WS_02 tại `WeakSup/propositions/v2/`.
5. Tạo `disease_alias_v2.csv`, che tên bệnh trong caption và tạo danh sách dấu hiệu cho LLM.
6. Dry-run, pilot có xác nhận riêng, rồi mới cho phép chạy **OpenAI Responses API** toàn bộ bằng cache/checkpoint bền vững.
7. Tùy chọn chạy lại ConceptCLIP độc lập để tạo checkpoint ảnh, hoặc tái sử dụng checkpoint đã có; sau đó ánh xạ theo `proposition_id`, hợp nhất và xuất nhãn.

> **Ranh giới WS_02 / WS_03:** artifact trong `WeakSup/propositions/v2/` là đầu vào **read-only**. Notebook này không export, tạo lại, sửa hoặc ghi đè bất kỳ file WS_02 hay file SkinCAP nào.
>
> **Cách ly phiên bản:** mọi đầu ra mới chỉ được ghi vào `WeakSup/weak_labels/v2/`. `weak_labels/v1/` và `disease_alias_v1.csv` không được đọc hoặc ghi bởi notebook này.
>
> **Bảo vệ LLM:** dry-run không gọi API. Pilot và full run mặc định bị khóa bởi các cờ `RUN_LLM_PILOT = False` và `RUN_LLM_FULL = False`; chỉ bật từng cờ sau khi đã xem bước trước.
>
> **Bảo vệ ConceptCLIP:** Mục 9a mặc định bị khóa bởi `RUN_CONCEPTCLIP = False`. Khi không chủ động chạy lại mô hình, chỉ copy checkpoint có sẵn vào `weak_labels/v2/checkpoints/`.
>
> **Bảo vệ bí mật:** `OPENAI_API_KEY` và `LLM_MODEL` chỉ được lấy từ Colab Secrets hoặc biến môi trường lúc chạy. Không lưu chúng, Hugging Face token, hay bất cứ key nào vào notebook, YAML, cache hoặc metadata.

## 1. Thiết lập môi trường Google Colab

Kiểm tra phiên bản Python, GPU khả dụng và thư mục làm việc hiện tại. **LLM-only workflow không cần GPU**; GPU chỉ bắt buộc nếu chủ động bật Mục 9a để chạy lại ConceptCLIP.

In [ ]:
import os
import platform
import sys

try:
    import torch
except ImportError as error:
    raise RuntimeError("Notebook này phải được chạy trong Google Colab có PyTorch.") from error

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Working directory: {os.getcwd()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA runtime: {torch.version.cuda}")
else:
    print("⚠️ Chưa có GPU. Bước gán nhãn mô hình nên chạy với GPU Colab.")

## 2. Cài đặt và nhập các thư viện

Cài bộ phụ thuộc tập trung cho SkinCAP trên Colab. Không dùng `requirements.txt` đầy đủ vì nó chứa nhiều gói phục vụ môi trường/nhánh khác, một số gói cần biên dịch và không phù hợp với Colab.

Gói SDK LLM chỉ được cài trong notebook và chỉ cần khi chạy pilot/full run. Nó không thay đổi requirements hoặc mã nguồn của RAD.

In [ ]:
# Các gói cơ bản cần cho các ô kiểm tra và đọc cấu hình.
# Các phụ thuộc SkinCAP/RAD sẽ được cài sau khi clone repository.
!pip -q install --no-cache-dir "PyYAML>=6.0" pandas numpy tqdm

import json
import subprocess
from pathlib import Path

import pandas as pd
import yaml

print("Đã import: pandas và PyYAML.")

## 3. Kết nối Google Drive và lấy branch `WS`

Drive cung cấp dữ liệu SkinCAP và nơi lưu artifact phiên bản. Repository được đặt tại `/content/RAD-colab`; ô này chỉ checkout branch `WS`, không tự tạo hay đổi các đường dẫn dữ liệu đã xác nhận.

In [ ]:
try:
    from google.colab import drive
except ImportError as error:
    raise RuntimeError("Hãy chạy notebook này trong Google Colab.") from error

drive.mount("/content/drive", force_remount=False)

REPO_URL = "https://github.com/trisle0805/RAD-colab.git"
REPO_DIR = Path("/content/RAD-colab")
BRANCH = "WS"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

current_branch = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True
).strip()
assert current_branch == BRANCH, f"Expected branch {BRANCH!r}, got {current_branch!r}"
print(f"Repository: {REPO_DIR}")
print(f"Branch: {current_branch}")

In [ ]:
# The repository has now been cloned, so install its focused Colab dependency set.
requirements_file = REPO_DIR / "requirements" / "colab-skin.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Missing Colab requirements file: {requirements_file}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "-r", str(requirements_file)],
    check=True,
)
print(f"Đã cài phụ thuộc từ: {requirements_file}")

## 4. Khai báo đường dẫn dữ liệu và kết quả v2

`configs/ws.yaml` vẫn là nguồn kiểm chứng cho CSV và thư mục ảnh SkinCAP. Vì YAML hiện lưu cấu hình v1, notebook chỉ dùng các phần đầu vào SkinCAP và tự suy ra các đường dẫn **v2** từ `ws_root`; không sửa YAML và không tham chiếu bất kỳ đầu ra v1 nào.

Artifact WS_02 trong `WeakSup/propositions/v2/` là **đầu vào chỉ đọc**.

In [ ]:
CONFIG_PATH = REPO_DIR / "configs" / "ws.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing WS config: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

assert Path(cfg["repo_dir"]) == REPO_DIR, (
    f"configs/ws.yaml repo_dir must be {REPO_DIR}, got {cfg['repo_dir']}"
)

# Keep ws.yaml read-only. v2 is deliberately derived here so the notebook has
# no operational dependency on the old v1 paths or BioClinicalBERT settings.
WS_ROOT = Path(cfg["ws_root"])
PROPOSITIONS_DIR = WS_ROOT / "propositions" / "v2"
OUTPUT_DIR = WS_ROOT / "weak_labels" / "v2"

PATHS = {
    "train_csv": Path(cfg["skincap"]["train_csv"]),
    "val_csv": Path(cfg["skincap"]["val_csv"]),
    "test_csv": Path(cfg["skincap"]["test_csv"]),
    "image_root": Path(cfg["skincap"]["image_root"]),
    "propositions_dir": PROPOSITIONS_DIR,
    "propositions_review": PROPOSITIONS_DIR / "propositions_review.csv",
    "propositions_final": PROPOSITIONS_DIR / "propositions_final.json",
    "propositions_summary": PROPOSITIONS_DIR / "summary_per_disease.csv",
    "propositions_meta": PROPOSITIONS_DIR / "meta.json",
    "weak_labels_dir": OUTPUT_DIR,
}

print("Cấu hình SkinCAP đã nạp từ:", CONFIG_PATH)
print("WS_03 v2 paths are notebook-local; configs/ws.yaml was not modified.")
for name, path in PATHS.items():
    print(f"- {name}: {path}")

In [ ]:
def require_file(label: str, path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"{label} không tồn tại: {path}")
    return path


def require_directory(label: str, path: Path) -> Path:
    if not path.is_dir():
        raise NotADirectoryError(f"{label} không tồn tại hoặc không phải thư mục: {path}")
    return path


for split in ("train_csv", "val_csv", "test_csv"):
    require_file(split, PATHS[split])
require_directory("image_root", PATHS["image_root"])

# WS_03 requires the complete, already-approved WS_02 version on Drive.
require_directory("propositions_dir", PATHS["propositions_dir"])
for artifact in (
    "propositions_review",
    "propositions_final",
    "propositions_summary",
    "propositions_meta",
):
    require_file(artifact, PATHS[artifact])

# Check split schema and case counts without copying data.
split_frames = {}
for split in ("train", "val", "test"):
    csv_path = PATHS[f"{split}_csv"]
    frame = pd.read_csv(csv_path)
    if frame.empty:
        raise ValueError(f"{split} split is empty: {csv_path}")
    split_frames[split] = frame
    print(f"{split:>5}: {len(frame):,} rows | columns: {list(frame.columns)}")

image_count = sum(1 for path in PATHS["image_root"].iterdir() if path.is_file())
print(f"Images directly in image_root: {image_count:,}")
print("WS_02 artifacts: present and read-only.")

## 5. Đọc artifact đã chốt của WS_02 v2

Bước 3 chỉ đọc `WeakSup/propositions/v2/propositions_final.json` để lấy danh sách proposition, hypothesis và cờ `image_applicable`. `propositions_review.csv`, `summary_per_disease.csv` và `meta.json` được kiểm tra để bảo đảm bộ artifact WS_02 đầy đủ và truy vết được. Notebook không gọi `export_final_version` và không ghi vào thư mục `WeakSup/propositions/v2/`.

In [ ]:
propositions_dir = PATHS["propositions_dir"]
propositions_final = PATHS["propositions_final"]

# Read-only: WS_02 has already produced and approved this version on Drive.
with propositions_final.open("r", encoding="utf-8") as handle:
    propositions = json.load(handle)

if not isinstance(propositions, list) or not propositions:
    raise ValueError(f"Invalid or empty propositions file: {propositions_final}")

with PATHS["propositions_meta"].open("r", encoding="utf-8") as handle:
    propositions_meta = json.load(handle)

summary = pd.read_csv(PATHS["propositions_summary"])
if summary.empty:
    raise ValueError(f"Empty WS_02 summary: {PATHS['propositions_summary']}")

print(f"WS_02 directory (read-only): {propositions_dir}")
print(f"Observable propositions: {len(propositions):,}")
print(f"Diseases in WS_02 summary: {len(summary):,}")
print(f"WS_02 stage: {propositions_meta.get('stage', 'unknown')}")
print("First proposition:", json.dumps(propositions[0], ensure_ascii=False, indent=2))

## 5.1 Điều kiện trước khi gán nhãn yếu v2

WS_03 v2 **không dùng BioClinicalBERT/NLI**. Text teacher là LLM, còn ConceptCLIP được tái sử dụng từ checkpoint đã chạy trước đó. Khi đến ô ConceptCLIP, hãy copy checkpoint vào đúng đường dẫn v2:

`WeakSup/weak_labels/v2/checkpoints/conceptclip_checkpoint.npz`

Notebook không tải hoặc chạy lại ConceptCLIP.

In [ ]:
IMAGE_MODEL_ID = str(cfg["image_labeler"]["model_id"])
IMAGE_POSITIVE_THRESHOLD = float(cfg["image_labeler"]["positive_threshold"])
IMAGE_NEGATIVE_THRESHOLD = float(cfg["image_labeler"]["negative_threshold"])

print("Text teacher: LLM (BioClinicalBERT/NLI is intentionally not used in v2).")
print(f"Optional image teacher: {IMAGE_MODEL_ID}")
print("ConceptCLIP can be run in optional Mục 9a, or its checkpoint can be copied before the merge step.")
print("Expected filename: checkpoints/conceptclip_checkpoint.npz")
print("Image thresholds:", {"positive": IMAGE_POSITIVE_THRESHOLD, "negative": IMAGE_NEGATIVE_THRESHOLD})

## 6. Chuẩn bị alias v2, caption đã che và hàm dùng chung

Ô này chỉ tạo/nạp `disease_alias_v2.csv` trong thư mục v2. File alias v1 không được sử dụng. Hãy duyệt/chỉnh alias v2 trên Drive trước khi chạy LLM.

Caption chỉ được che tên bệnh cho bộ gán nhãn; mô hình RAD ở các bước sau vẫn dùng caption gốc.

In [ ]:
import hashlib
from collections import defaultdict
from datetime import datetime, timezone

import numpy as np
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_DIR))
from proposition.weak.labels import (
    ABSENT,
    PRESENT,
    UNKNOWN,
    load_propositions,
    mask_disease_aliases,
    merge_labels,
    normalise_whitespace,
    quote_is_supported,
    read_alias_csv,
    suggested_alias_rows,
    write_alias_csv,
)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
ALIAS_CSV = OUTPUT_DIR / "disease_alias_v2.csv"
CAPTIONS_MASKED_PATH = OUTPUT_DIR / "captions_masked.json"
SIGN_MAP_PATH = OUTPUT_DIR / "sign_map.json"
LLM_CACHE_PATH = OUTPUT_DIR / "llm_cache.jsonl"
LLM_DRY_RUN_PATH = OUTPUT_DIR / "llm_dry_run_report.csv"
LLM_CHECKPOINT = CHECKPOINT_DIR / "llm_checkpoint.npz"
IMAGE_CHECKPOINT = CHECKPOINT_DIR / "conceptclip_checkpoint.npz"
LABELS_PATH = OUTPUT_DIR / "labels.npz"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not ALIAS_CSV.exists():
    disease_columns = list(split_frames["train"].columns[2:])
    write_alias_csv(ALIAS_CSV, suggested_alias_rows(disease_columns))

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_cases() -> tuple[list[dict[str, str]], list[str]]:
    cases: list[dict[str, str]] = []
    disease_columns: list[str] = []
    for split in ("train", "val", "test"):
        frame = pd.read_csv(PATHS[f"{split}_csv"], dtype=str).fillna("")
        case_column, caption_column = frame.columns[:2]
        if split == "train":
            disease_columns = list(frame.columns[2:])
        for _, item in frame.iterrows():
            positives = [
                name for name in disease_columns
                if name in item and str(item[name]) in {"1", "1.0", "True", "true"}
            ]
            cases.append({
                "case_id": str(item[case_column]),
                "split": split,
                "caption": str(item[caption_column]),
                "disease": positives[0] if len(positives) == 1 else "",
            })
    if len({case["case_id"] for case in cases}) != len(cases):
        raise ValueError("case_id phải là duy nhất trên train/val/test.")
    return cases, disease_columns

PROPOSITIONS = load_propositions(PATHS["propositions_final"], PATHS["propositions_review"])
CASES, DISEASE_COLUMNS = load_cases()
ALIASES = read_alias_csv(ALIAS_CSV)
CAPTIONS_MASKED = {
    case["case_id"]: mask_disease_aliases(case["caption"], ALIASES)
    for case in CASES
}
CAPTIONS_MASKED_PATH.write_text(
    json.dumps(CAPTIONS_MASKED, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("Alias CSV v2:", ALIAS_CSV)
display(pd.read_csv(ALIAS_CSV).head(20))
print(f"Cases: {len(CASES):,}; propositions v2: {len(PROPOSITIONS):,}")
print("LLM checkpoint:", LLM_CHECKPOINT)
print("ConceptCLIP checkpoint to reuse:", IMAGE_CHECKPOINT)
print("Duyệt/chỉnh disease_alias_v2.csv trên Drive trước khi chạy LLM.")

## 7. LLM text labeler: danh sách dấu hiệu và dry-run bắt buộc

LLM thay thế hoàn toàn BioClinicalBERT/NLI. Các mệnh đề được gộp theo `sub_hypothesis` đã chuẩn hoá trong bốn nhóm `morphology`, `location`, `symptom`, `history`.

Ô dry-run tiếp theo **không gọi API**. Nó tạo `sign_map.json`, ước lượng số lời gọi/token/chi phí và ghi `llm_dry_run_report.csv`. Hãy xem kết quả rồi mới bật pilot.

In [ ]:
# Không cần Hugging Face Secret khi chỉ copy/nạp checkpoint ConceptCLIP đã có.
# Chỉ khi RUN_CONCEPTCLIP=True mà Hugging Face yêu cầu xác thực, thêm Secret HF_TOKEN
# và cấp quyền notebook access trong giao diện Secrets của Colab.
print("Không cần Hugging Face Secret để dùng checkpoint ConceptCLIP đã copy.")

In [ ]:
# OpenAI SDK mới nhất cho Responses API (ví dụ model gpt-6-luna).
# Không pin phiên bản để notebook dùng SDK hiện hành trên Colab.
!pip -q install --no-cache-dir --upgrade openai

In [ ]:
CATEGORIES = ("morphology", "location", "symptom", "history")
PROMPT_TEMPLATE = """You are extracting what a clinical description of a skin case explicitly states about the patient.

You will receive a list of signs and then one description. For each sign, decide:
- PRESENT: the description explicitly states that this patient has the sign. If a sign lists alternatives separated by " / ", one is enough.
- ABSENT: the description explicitly states that this patient does NOT have the sign, or explicitly describes the same feature in a directly opposite way.
- NOT_MENTIONED: anything else. This is the default.

Rules:
1. Use ONLY the words of the description. Do NOT use medical knowledge to infer signs that are not written.
2. Do NOT guess the diagnosis and do not mark a sign because it is typical of a suspected disease.
3. Only statements about THIS patient count. Generic condition descriptions are NOT evidence.
4. Sharing words is not enough; respect sites, attributes, and the scope of negations.
5. When unsure, answer NOT_MENTIONED.
6. For PRESENT or ABSENT, copy a short exact supporting phrase from the description.

Signs ({category}):
{sign_lines}

Return JSON only. List ONLY the signs that are PRESENT or ABSENT; omit NOT_MENTIONED signs:
{{"results": [{{"id": "S001", "verdict": "PRESENT|ABSENT", "quote": "<verbatim phrase>"}}]}}

Description:
<<<{masked_caption}>>>"""
PROMPT_VERSION = hashlib.sha256(PROMPT_TEMPLATE.encode("utf-8")).hexdigest()

# A sign is a normalised sub-hypothesis shared by every proposition that uses it.
by_category: dict[str, dict[str, dict[str, object]]] = {category: {} for category in CATEGORIES}
for proposition in PROPOSITIONS:
    if proposition.category not in by_category:
        raise ValueError(f"Unsupported proposition category: {proposition.category}")
    for sub_hypothesis in proposition.sub_hypotheses:
        key = normalise_whitespace(sub_hypothesis)
        if not key:
            raise ValueError(f"Empty sub-hypothesis in {proposition.proposition_id}")
        item = by_category[proposition.category].setdefault(
            key,
            {"text_parts": [], "proposition_ids": []},
        )
        if sub_hypothesis not in item["text_parts"]:
            item["text_parts"].append(sub_hypothesis)
        if proposition.proposition_id not in item["proposition_ids"]:
            item["proposition_ids"].append(proposition.proposition_id)

SIGN_MAP: dict[str, dict[str, dict[str, object]]] = {category: {} for category in CATEGORIES}
sign_number = 1
for category in CATEGORIES:
    for key in sorted(by_category[category]):
        item = by_category[category][key]
        sign_id = f"S{sign_number:03d}"
        SIGN_MAP[category][sign_id] = {
            "text": " / ".join(item["text_parts"]),
            "proposition_ids": sorted(item["proposition_ids"]),
        }
        sign_number += 1

SIGN_MAP_PATH.write_text(json.dumps(SIGN_MAP, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
PROP_COLUMN = {prop.proposition_id: column for column, prop in enumerate(PROPOSITIONS)}
SIGN_LINES = {
    category: "\n".join(f"{sign_id} | {item['text']}" for sign_id, item in SIGN_MAP[category].items())
    for category in CATEGORIES
}

def render_prompt(category: str, masked_caption: str) -> str:
    return PROMPT_TEMPLATE.format(
        category=category,
        sign_lines=SIGN_LINES[category],
        masked_caption=masked_caption,
    )

def cache_key(case_id: str, category: str, model: str) -> tuple[str, str, str, str]:
    return case_id, category, PROMPT_VERSION, model

def read_cache() -> dict[tuple[str, str, str, str], dict[str, object]]:
    records: dict[tuple[str, str, str, str], dict[str, object]] = {}
    if not LLM_CACHE_PATH.is_file():
        return records
    with LLM_CACHE_PATH.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
                key = cache_key(
                    str(record["case_id"]), str(record["category"]), str(record["model"])
                )
            except (json.JSONDecodeError, KeyError, TypeError) as error:
                raise ValueError(f"Invalid JSONL cache record at line {line_number}") from error
            if record.get("prompt_version") == PROMPT_VERSION:
                records[key] = record
    return records

def append_cache(record: dict[str, object]) -> None:
    with LLM_CACHE_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()

def parse_llm_reply(raw_text: str, category: str, masked_caption: str) -> tuple[list[dict[str, object]], int]:
    """Return validated non-unknown results and count JSON/schema errors."""
    text = str(raw_text).strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    try:
        payload = json.loads(text)
        values = payload["results"]
        if not isinstance(values, list):
            raise ValueError("results is not a list")
    except (json.JSONDecodeError, KeyError, TypeError, ValueError):
        return [], 1

    grouped: dict[str, list[tuple[str, str, bool]]] = defaultdict(list)
    known_ids = set(SIGN_MAP[category])
    for value in values:
        if not isinstance(value, dict):
            continue
        sign_id = str(value.get("id", ""))
        verdict = str(value.get("verdict", "")).upper()
        quote = str(value.get("quote", ""))
        if sign_id not in known_ids or verdict not in {"PRESENT", "ABSENT"}:
            continue
        grouped[sign_id].append((verdict, quote, quote_is_supported(quote, masked_caption)))

    cleaned: list[dict[str, object]] = []
    for sign_id, answers in grouped.items():
        verdicts = {answer[0] for answer in answers}
        invalid_quote = any(not answer[2] for answer in answers)
        if len(verdicts) != 1 or invalid_quote:
            cleaned.append({"id": sign_id, "verdict": "NOT_MENTIONED", "quote": "", "quote_valid": not invalid_quote})
            continue
        verdict, quote, _ = answers[0]
        cleaned.append({"id": sign_id, "verdict": verdict, "quote": quote, "quote_valid": True})
    return cleaned, 0

def get_runtime_value(name: str) -> str | None:
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return str(value) if value else None
    except Exception:
        return None

def llm_runtime() -> tuple[str, str]:
    model = (get_runtime_value("LLM_MODEL") or "").strip()
    api_key = get_runtime_value("OPENAI_API_KEY")
    if not model:
        raise RuntimeError("Set Colab Secret/environment LLM_MODEL before a real LLM run.")
    if not api_key:
        raise RuntimeError("Set Colab Secret/environment OPENAI_API_KEY; do not put it in this notebook.")
    return model, api_key

def invoke_llm(model: str, api_key: str, prompt: str) -> tuple[str, dict[str, int]]:
    """Call the OpenAI Responses API with the current Colab Secret values."""
    from openai import OpenAI

    client = OpenAI(api_key=api_key)
    response = client.responses.create(
        model=model,
        input=prompt,
    )
    usage = getattr(response, "usage", None)
    return response.output_text or "", {
        "input_tokens": int(getattr(usage, "input_tokens", 0) or 0),
        "output_tokens": int(getattr(usage, "output_tokens", 0) or 0),
        "cache_tokens": int(getattr(getattr(usage, "input_tokens_details", None), "cached_tokens", 0) or 0),
    }

def materialize_llm_labels(model: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict[tuple[str, str], dict[str, object]]]:
    labels = np.full((len(CASES), len(PROPOSITIONS)), UNKNOWN, dtype=np.int8)
    quote_invalid = np.zeros(labels.shape, dtype=np.int8)
    completed = np.zeros((len(CASES), len(CATEGORIES)), dtype=np.int8)
    evidence: dict[tuple[str, str], dict[str, object]] = {}
    records = read_cache()
    for row, case in enumerate(CASES):
        for category_index, category in enumerate(CATEGORIES):
            record = records.get(cache_key(case["case_id"], category, model))
            if record is None:
                continue
            completed[row, category_index] = 1
            for result in record.get("results", []):
                sign_id = str(result.get("id", ""))
                if sign_id not in SIGN_MAP[category]:
                    continue
                quote_valid = bool(result.get("quote_valid", False))
                verdict = str(result.get("verdict", "NOT_MENTIONED"))
                value = PRESENT if verdict == "PRESENT" and quote_valid else ABSENT if verdict == "ABSENT" and quote_valid else UNKNOWN
                for proposition_id in SIGN_MAP[category][sign_id]["proposition_ids"]:
                    column = PROP_COLUMN[proposition_id]
                    labels[row, column] = value
                    if not quote_valid:
                        quote_invalid[row, column] = 1
                    evidence[(case["case_id"], proposition_id)] = {
                        "verdict": verdict if value != UNKNOWN else "NOT_MENTIONED",
                        "quote": str(result.get("quote", "")) if quote_valid else "",
                        "quote_valid": int(quote_valid),
                    }
    return labels, quote_invalid, completed, evidence

def write_llm_checkpoint(model: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict[tuple[str, str], dict[str, object]]]:
    labels, quote_invalid, completed, evidence = materialize_llm_labels(model)
    np.savez_compressed(
        LLM_CHECKPOINT,
        case_id=np.asarray([case["case_id"] for case in CASES]),
        split=np.asarray([case["split"] for case in CASES]),
        proposition_id=np.asarray([prop.proposition_id for prop in PROPOSITIONS]),
        label_text=labels,
        llm_quote_invalid=quote_invalid,
        completed_categories=completed,
    )
    return labels, quote_invalid, completed, evidence

def run_llm_cases(cases_to_run: list[dict[str, str]], run_name: str) -> dict[str, int]:
    model, api_key = llm_runtime()
    records = read_cache()
    totals = {"new_calls": 0, "input_tokens": 0, "cache_tokens": 0, "output_tokens": 0, "json_errors": 0, "api_errors": 0}
    for case in tqdm(cases_to_run, desc=f"LLM {run_name}"):
        for category in CATEGORIES:
            key = cache_key(case["case_id"], category, model)
            if key in records:
                continue
            prompt = render_prompt(category, CAPTIONS_MASKED[case["case_id"]])
            last_error = ""
            for attempt in range(1, 4):
                try:
                    raw_response, usage = invoke_llm(model, api_key, prompt)
                    results, json_errors = parse_llm_reply(raw_response, category, CAPTIONS_MASKED[case["case_id"]])
                    totals["json_errors"] += json_errors
                    if json_errors:
                        raise ValueError("LLM response is not valid schema-conforming JSON")
                    record = {
                        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                        "status": "ok", "run": run_name, "case_id": case["case_id"], "category": category,
                        "provider": "openai_responses", "model": model, "prompt_version": PROMPT_VERSION,
                        "attempt": attempt, "usage": usage, "results": results, "raw_response": raw_response,
                    }
                    append_cache(record)
                    records[key] = record
                    totals["new_calls"] += 1
                    for name in ("input_tokens", "cache_tokens", "output_tokens"):
                        totals[name] += int(usage.get(name, 0))
                    break
                except Exception as error:
                    last_error = f"{type(error).__name__}: {error}"
                    if attempt == 3:
                        record = {
                            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                            "status": "error", "run": run_name, "case_id": case["case_id"], "category": category,
                            "provider": "openai_responses", "model": model, "prompt_version": PROMPT_VERSION,
                            "attempt": attempt, "error": last_error, "results": [],
                        }
                        append_cache(record)
                        records[key] = record
                        totals["api_errors"] += 1
            write_llm_checkpoint(model)
    return totals

# Required dry-run: no API key is read and no network call is made.
dry_rows = []
for category in CATEGORIES:
    static_tokens = len(render_prompt(category, "")) // 4
    caption_tokens = sum(len(CAPTIONS_MASKED[case["case_id"]]) // 4 for case in CASES)
    dry_rows.append({
        "category": category,
        "n_signs": len(SIGN_MAP[category]),
        "n_cases": len(CASES),
        "estimated_calls": len(CASES),
        "estimated_cached_input_tokens": len(CASES) * static_tokens,
        "estimated_caption_input_tokens": caption_tokens,
        "estimated_output_tokens": len(CASES) * max(32, len(SIGN_MAP[category]) * 8),
    })
dry_report = pd.DataFrame(dry_rows)
input_price = float(os.environ.get("LLM_INPUT_PRICE_PER_MTOK", "nan"))
output_price = float(os.environ.get("LLM_OUTPUT_PRICE_PER_MTOK", "nan"))
if np.isfinite(input_price) and np.isfinite(output_price):
    dry_report["estimated_cost"] = (
        (dry_report["estimated_cached_input_tokens"] + dry_report["estimated_caption_input_tokens"]) * input_price
        + dry_report["estimated_output_tokens"] * output_price
    ) / 1_000_000
dry_report.to_csv(LLM_DRY_RUN_PATH, index=False)

print("Prompt version:", PROMPT_VERSION)
print("Sign map:", SIGN_MAP_PATH)
display(dry_report)
print("Đây là dry-run: chưa đọc API key và chưa gọi LLM. Dừng tại đây để xem chi phí trước khi chạy pilot.")

## 8. Pilot LLM: 47 ca validation có xác nhận riêng

Sau khi xem dry-run, tạo Colab Secrets `OPENAI_API_KEY` và `LLM_MODEL` (ví dụ `gpt-6-luna`), rồi đặt `RUN_LLM_PILOT = True` trong ô dưới để gọi OpenAI Responses API cho một ca validation của mỗi bệnh, với seed 42. Ô sẽ ghi cache/checkpoint sau mỗi ca và xuất `llm_pilot/pilot.csv` để kiểm tra nhãn trên ca cùng bệnh và bệnh khác.

Không bật full run trước khi đã xem bảng pilot.

In [ ]:
RUN_LLM_PILOT = False  # Chỉ đổi thành True sau khi đã xem llm_dry_run_report.csv.
PILOT_DIR = OUTPUT_DIR / "llm_pilot"
PILOT_PATH = PILOT_DIR / "pilot.csv"

val_cases = [case for case in CASES if case["split"] == "val" and case["disease"]]
rng = np.random.default_rng(42)
pilot_cases: list[dict[str, str]] = []
for disease_id in sorted({case["disease"] for case in val_cases}):
    options = [case for case in val_cases if case["disease"] == disease_id]
    pilot_cases.append(options[int(rng.integers(len(options)))])

print(f"Pilot selection: {len(pilot_cases)} validation cases, one per disease (seed=42).")
if RUN_LLM_PILOT:
    pilot_totals = run_llm_cases(pilot_cases, "pilot")
    _, _, _, evidence = write_llm_checkpoint((get_runtime_value("LLM_MODEL") or "").strip())
    rows = []
    for case in pilot_cases:
        for proposition in PROPOSITIONS:
            item = evidence.get((case["case_id"], proposition.proposition_id), {})
            rows.append({
                "case_id": case["case_id"],
                "disease_id": case["disease"],
                "caption_masked": CAPTIONS_MASKED[case["case_id"]],
                "proposition_id": proposition.proposition_id,
                "canonicalDescription": proposition.canonical_description,
                "category": proposition.category,
                "verdict": item.get("verdict", "NOT_MENTIONED"),
                "quote": item.get("quote", ""),
                "quote_valid": item.get("quote_valid", 0),
                "same_disease": int(case["disease"] == proposition.disease_id),
            })
    PILOT_DIR.mkdir(parents=True, exist_ok=True)
    pilot_frame = pd.DataFrame(rows)
    pilot_frame.to_csv(PILOT_PATH, index=False)
    present_summary = (
        pilot_frame.assign(is_present=pilot_frame["verdict"].eq("PRESENT"))
        .groupby("same_disease", as_index=False)["is_present"]
        .mean()
        .rename(columns={"is_present": "present_rate"})
    )
    print("Pilot totals:", pilot_totals)
    display(present_summary)
    print("Đã lưu pilot:", PILOT_PATH)
else:
    print("Pilot bị khóa. Đổi RUN_LLM_PILOT=True chỉ sau khi đã xác nhận dry-run.")

## 9. Chạy LLM đầy đủ

Chỉ đặt `RUN_LLM_FULL = True` sau khi đã xem `llm_pilot/pilot.csv`. Ô này **chỉ chạy/gom kết quả LLM**: khôi phục từ `llm_cache.jsonl`, ghi `llm_checkpoint.npz` sau mỗi ca và không gọi lại nhóm đã cache.

Ô không nạp, không chạy, và không hợp nhất ConceptCLIP. Hãy chạy ô 9b riêng sau khi LLM hoàn tất.

In [ ]:
RUN_LLM_FULL = False  # Chỉ đổi thành True sau khi đã xem và chấp nhận kết quả pilot.

if RUN_LLM_FULL:
    full_totals = run_llm_cases(CASES, "full")
    print("Full-run totals:", full_totals)
    model_for_checkpoint = (get_runtime_value("LLM_MODEL") or "").strip()
    label_text, llm_quote_invalid, completed_categories, _ = write_llm_checkpoint(model_for_checkpoint)
    missing_groups = int(np.count_nonzero(completed_categories == 0))
    completed_groups = int(np.count_nonzero(completed_categories))
    print(
        f"LLM checkpoint: {LLM_CHECKPOINT}\n"
        f"Completed case-category groups: {completed_groups}/{completed_categories.size}; "
        f"missing: {missing_groups}"
    )
    if missing_groups:
        print("LLM chưa hoàn tất. Không chạy ô hợp nhất LLM + ConceptCLIP cho đến khi missing=0.")
    else:
        print("LLM đã hoàn tất. Có thể chạy ô 9b để hợp nhất với ConceptCLIP checkpoint.")
else:
    print("Full run bị khóa. Bật RUN_LLM_FULL=True chỉ sau khi đã duyệt pilot.")
    print("Ô này không cần LLM Secret khi đang khóa.")

## 9a. Tùy chọn: chạy ConceptCLIP độc lập

Ô này giữ đầy đủ code chạy ConceptCLIP để tái tạo checkpoint ảnh độc lập khi cần. Nó không dùng LLM, không hợp nhất nhãn, và không sửa kết quả WS_02/v1.

Mặc định `RUN_CONCEPTCLIP = False`. Với kế hoạch hiện tại, **không chạy ô này**; chỉ copy checkpoint đã có vào `checkpoints/conceptclip_checkpoint.npz`. Nếu chủ động chạy lại, cần GPU Colab và model sẽ được tải từ Hugging Face `JerrryNie/ConceptCLIP`.

In [ ]:
RUN_CONCEPTCLIP = False  # Giữ False để dùng checkpoint đã có; chỉ bật khi cần tái tạo image checkpoint.

if not RUN_CONCEPTCLIP:
    print(
        "ConceptCLIP bị khóa. Dùng checkpoint đã có tại:\n"
        f"{IMAGE_CHECKPOINT}\n"
        "Chỉ đổi RUN_CONCEPTCLIP=True khi chủ động tái chạy image teacher trên GPU Colab."
    )
else:
    if not torch.cuda.is_available():
        raise RuntimeError("ConceptCLIP phải chạy trên GPU Colab. Hãy chọn GPU runtime rồi chạy lại.")

    from PIL import Image
    from transformers import AutoModel, AutoProcessor

    device = torch.device("cuda")
    hf_token = get_runtime_value("HF_TOKEN")
    print(f"Đang nạp {IMAGE_MODEL_ID} trên {torch.cuda.get_device_name(0)}...")
    model = AutoModel.from_pretrained(
        IMAGE_MODEL_ID,
        trust_remote_code=True,
        token=hf_token or None,
    ).to(device).eval()
    processor = AutoProcessor.from_pretrained(
        IMAGE_MODEL_ID,
        trust_remote_code=True,
        token=hf_token or None,
    )

    applicable = [
        (column, proposition)
        for column, proposition in enumerate(PROPOSITIONS)
        if proposition.image_applicable
    ]
    image_prompts = [
        prompt
        for _, proposition in applicable
        for sub_hypothesis in proposition.sub_hypotheses
        for prompt in (
            f"a clinical photo of skin with {sub_hypothesis}",
            f"this skin lesion shows {sub_hypothesis}",
            f"a clinical photo of skin without {sub_hypothesis}",
            f"this skin lesion does not show {sub_hypothesis}",
        )
    ]
    try:
        tokenized_prompts = processor(
            text=image_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        tokenized_prompts = {key: value.to(device) for key, value in tokenized_prompts.items()}
        with torch.inference_mode():
            text_features = model.get_text_features(**tokenized_prompts)
    except AttributeError as error:
        raise RuntimeError(
            "ConceptCLIP remote code lacks get_text_features; inspect its Hugging Face API before running."
        ) from error
    text_features = torch.nn.functional.normalize(text_features.float(), dim=-1)

    conceptclip_label_img = np.full((len(CASES), len(PROPOSITIONS)), UNKNOWN, dtype=np.int8)
    conceptclip_p_img = np.full((len(CASES), len(PROPOSITIONS)), np.nan, dtype=np.float16)
    image_root = PATHS["image_root"]

    for row, case in enumerate(tqdm(CASES, desc="WS_03 ConceptCLIP")):
        image_path = Path(case["case_id"])
        image_path = image_path if image_path.is_absolute() else image_root / image_path
        if not image_path.is_file():
            raise FileNotFoundError(f"Image for {case['case_id']} is missing: {image_path}")
        try:
            with Image.open(image_path) as image:
                image_inputs = processor(images=image.convert("RGB"), return_tensors="pt")
            image_inputs = {key: value.to(device) for key, value in image_inputs.items()}
            with torch.inference_mode():
                image_feature = model.get_image_features(**image_inputs)
        except AttributeError as error:
            raise RuntimeError(
                "ConceptCLIP remote code lacks get_image_features; inspect its Hugging Face API before running."
            ) from error
        image_feature = torch.nn.functional.normalize(image_feature.float(), dim=-1)[0]

        offset = 0
        for column, proposition in applicable:
            hypothesis_scores: list[float] = []
            for _ in proposition.sub_hypotheses:
                positive_score = (image_feature @ text_features[offset:offset + 2].T).mean()
                negative_score = (image_feature @ text_features[offset + 2:offset + 4].T).mean()
                logit_scale = getattr(model, "logit_scale", torch.tensor(1.0, device=device))
                scale = logit_scale.exp() if torch.is_tensor(logit_scale) else 1.0
                probability = torch.softmax(scale * torch.stack((positive_score, negative_score)), dim=0)[0]
                hypothesis_scores.append(float(probability))
                offset += 4
            score = max(hypothesis_scores)
            conceptclip_p_img[row, column] = score
            if score >= IMAGE_POSITIVE_THRESHOLD:
                conceptclip_label_img[row, column] = PRESENT
            elif all(value <= IMAGE_NEGATIVE_THRESHOLD for value in hypothesis_scores):
                conceptclip_label_img[row, column] = ABSENT

    np.savez_compressed(
        IMAGE_CHECKPOINT,
        case_id=np.asarray([case["case_id"] for case in CASES]),
        split=np.asarray([case["split"] for case in CASES]),
        proposition_id=np.asarray([proposition.proposition_id for proposition in PROPOSITIONS]),
        label_img=conceptclip_label_img,
        p_img=conceptclip_p_img,
        model_id=np.asarray(IMAGE_MODEL_ID),
        positive_threshold=np.asarray(IMAGE_POSITIVE_THRESHOLD),
        negative_threshold=np.asarray(IMAGE_NEGATIVE_THRESHOLD),
    )
    print("Đã lưu ConceptCLIP checkpoint:", IMAGE_CHECKPOINT)
    print(
        "Image labels: present=%d, absent=%d, unknown=%d" % (
            np.count_nonzero(conceptclip_label_img == PRESENT),
            np.count_nonzero(conceptclip_label_img == ABSENT),
            np.count_nonzero(conceptclip_label_img == UNKNOWN),
        )
    )

## 9b. Hợp nhất checkpoint LLM và ConceptCLIP

Ô này **không chạy ConceptCLIP**. Nó chỉ đọc `conceptclip_checkpoint.npz` đã được tạo bằng ô 9a hoặc copy riêng vào thư mục v2, ánh xạ `p_img` theo `case_id` và `proposition_id`, sau đó hợp nhất với `llm_checkpoint.npz` để tạo `labels.npz` và `meta.json`.

Vì checkpoint ConceptCLIP phải có sẵn, chạy ô này chỉ khi Mục 9 báo `missing: 0` và file checkpoint ảnh đã hiện diện.

In [ ]:
model_for_checkpoint = (get_runtime_value("LLM_MODEL") or "").strip()
if not model_for_checkpoint:
    raise RuntimeError("Set LLM_MODEL to select the matching cached LLM results for merging.")

if not LLM_CHECKPOINT.is_file():
    raise FileNotFoundError(
        "LLM checkpoint chưa có. Hoàn tất Mục 9 trước khi hợp nhất: "
        f"{LLM_CHECKPOINT}"
    )

with np.load(LLM_CHECKPOINT, allow_pickle=False) as archive:
    required = {"case_id", "proposition_id", "label_text", "llm_quote_invalid", "completed_categories"}
    missing = required - set(archive.files)
    if missing:
        raise ValueError(f"LLM checkpoint thiếu arrays: {sorted(missing)}")
    llm_case_ids = [str(value) for value in archive["case_id"].tolist()]
    llm_prop_ids = [str(value) for value in archive["proposition_id"].tolist()]
    label_text = archive["label_text"]
    llm_quote_invalid = archive["llm_quote_invalid"]
    completed_categories = archive["completed_categories"]

expected_case_ids = [case["case_id"] for case in CASES]
expected_prop_ids = [proposition.proposition_id for proposition in PROPOSITIONS]
if llm_case_ids != expected_case_ids or llm_prop_ids != expected_prop_ids:
    raise ValueError("LLM checkpoint does not match the current case/proposition ordering.")
if label_text.shape != (len(CASES), len(PROPOSITIONS)):
    raise ValueError("LLM label_text shape does not match current cases/propositions.")
if llm_quote_invalid.shape != label_text.shape:
    raise ValueError("LLM llm_quote_invalid shape does not match label_text.")
if completed_categories.shape != (len(CASES), len(CATEGORIES)):
    raise ValueError("LLM completed_categories shape does not match current cases/categories.")
if not np.all(completed_categories):
    missing_groups = int(np.count_nonzero(completed_categories == 0))
    raise RuntimeError(
        f"LLM labeling is incomplete ({missing_groups} case-category groups missing). "
        "Finish the full LLM run before final merge."
    )

if not IMAGE_CHECKPOINT.is_file():
    raise FileNotFoundError(
        "Copy the existing ConceptCLIP checkpoint to this exact v2 path before merging: "
        f"{IMAGE_CHECKPOINT}"
    )

with np.load(IMAGE_CHECKPOINT, allow_pickle=False) as archive:
    required = {"case_id", "proposition_id", "p_img"}
    missing = required - set(archive.files)
    if missing:
        raise ValueError(f"ConceptCLIP checkpoint thiếu arrays: {sorted(missing)}")
    source_case_ids = [str(value) for value in archive["case_id"].tolist()]
    source_prop_ids = [str(value) for value in archive["proposition_id"].tolist()]
    source_p_img = archive["p_img"]

if source_p_img.shape != (len(source_case_ids), len(source_prop_ids)):
    raise ValueError("p_img shape does not match checkpoint case/proposition axes.")
source_case_index = {case_id: row for row, case_id in enumerate(source_case_ids)}
source_prop_index = {prop_id: column for column, prop_id in enumerate(source_prop_ids)}
missing_cases = [case["case_id"] for case in CASES if case["case_id"] not in source_case_index]
if missing_cases:
    raise ValueError(f"ConceptCLIP checkpoint misses {len(missing_cases)} current cases; first: {missing_cases[:3]}")

label_img = np.full(label_text.shape, UNKNOWN, dtype=np.int8)
p_img = np.full(label_text.shape, np.nan, dtype=np.float16)
for row, case in enumerate(CASES):
    source_row = source_case_index[case["case_id"]]
    for column, proposition in enumerate(PROPOSITIONS):
        if not proposition.image_applicable:
            continue
        source_column = source_prop_index.get(proposition.proposition_id)
        if source_column is None:
            continue
        score = float(source_p_img[source_row, source_column])
        if not np.isfinite(score):
            continue
        p_img[row, column] = score
        if score >= IMAGE_POSITIVE_THRESHOLD:
            label_img[row, column] = PRESENT
        elif score <= IMAGE_NEGATIVE_THRESHOLD:
            label_img[row, column] = ABSENT

label_final = merge_labels(label_text, label_img, teacher="both")
np.savez_compressed(
    LABELS_PATH,
    case_id=np.asarray([case["case_id"] for case in CASES]),
    split=np.asarray([case["split"] for case in CASES]),
    proposition_id=np.asarray([prop.proposition_id for prop in PROPOSITIONS]),
    label_text=label_text,
    llm_quote_invalid=llm_quote_invalid,
    label_img=label_img,
    label_final=label_final,
    p_img=p_img,
)

records = read_cache()
matching_records = [record for record in records.values() if str(record.get("model")) == model_for_checkpoint]
usage_totals = {
    name: sum(int(record.get("usage", {}).get(name, 0) or 0) for record in matching_records)
    for name in ("input_tokens", "cache_tokens", "output_tokens")
}
meta = {
    "stage": "WS_03_weak_labels_v2",
    "teacher": "both",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": 42,
    "n_cases": len(CASES),
    "n_propositions": len(PROPOSITIONS),
    "case_disease": [{"case_id": case["case_id"], "disease_id": case["disease"]} for case in CASES],
    "propositions_final_sha256": sha256_file(PATHS["propositions_final"]),
    "propositions_review_sha256": sha256_file(PATHS["propositions_review"]),
    "alias_sha256": sha256_file(ALIAS_CSV),
    "llm": {
        "model": model_for_checkpoint,
        "prompt_version": PROMPT_VERSION,
        "temperature": 0,
        "cache_path": str(LLM_CACHE_PATH),
        "n_cache_records": len(matching_records),
        "n_error_records": sum(record.get("status") == "error" for record in matching_records),
        "usage": usage_totals,
    },
    "image": {
        "checkpoint": str(IMAGE_CHECKPOINT),
        "source_propositions": len(source_prop_ids),
        "mapped_propositions": int(sum(prop.proposition_id in source_prop_index for prop in PROPOSITIONS)),
        "positive_threshold": IMAGE_POSITIVE_THRESHOLD,
        "negative_threshold": IMAGE_NEGATIVE_THRESHOLD,
    },
    "artifacts": {
        "captions_masked": str(CAPTIONS_MASKED_PATH),
        "sign_map": str(SIGN_MAP_PATH),
        "llm_checkpoint": str(LLM_CHECKPOINT),
        "labels": str(LABELS_PATH),
    },
}
(OUTPUT_DIR / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Đã lưu:", LABELS_PATH)
print("Đã lưu:", OUTPUT_DIR / "meta.json")
print("Nhãn cuối: present=%d, absent=%d, unknown=%d" % (
    np.count_nonzero(label_final == PRESENT),
    np.count_nonzero(label_final == ABSENT),
    np.count_nonzero(label_final == UNKNOWN),
))

## 10. Xem khảo sát cơ bản của artifact v2

Ô này chỉ đọc `labels.npz`, `llm_cache.jsonl` và `meta.json` đã sinh. Khảo sát chi tiết/min-label/audit vẫn nằm ở notebook `WS_03b_label_survey.ipynb`, notebook đó cần được cập nhật riêng để đọc contract v2 không có các mảng NLI cũ.

In [ ]:
if not LABELS_PATH.is_file():
    raise FileNotFoundError(f"Chưa có artifact v2: {LABELS_PATH}")

with np.load(LABELS_PATH, allow_pickle=False) as archive:
    labels_preview = {key: archive[key] for key in archive.files}

rows = []
for source, values in (
    ("text", labels_preview["label_text"]),
    ("image", labels_preview["label_img"]),
    ("final", labels_preview["label_final"]),
):
    for split in ("all", "train", "val", "test"):
        mask = np.ones(len(labels_preview["split"]), dtype=bool) if split == "all" else labels_preview["split"].astype(str) == split
        selected = values[mask]
        rows.append({
            "source": source,
            "split": split,
            "present": int(np.count_nonzero(selected == PRESENT)),
            "absent": int(np.count_nonzero(selected == ABSENT)),
            "unknown": int(np.count_nonzero(selected == UNKNOWN)),
        })

display(pd.DataFrame(rows))
print(json.dumps(json.loads((OUTPUT_DIR / "meta.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=2))